## Web Search Agent

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true" 
os.environ['LANGSMITH_PROJECT'] = 'AgenticAITraining' 

In [6]:
from pprint import pprint

In [9]:
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(
    max_results = 5, 
    topic = 'general'
)

results = tavily_search_tool.invoke({'query': 'What happened at the last ipl match'})

pprint(results)

{'answer': None,
 'follow_up_questions': None,
 'images': [],
 'query': 'What happened at the last ipl match',
 'request_id': '2b87995f-534d-4571-8e12-38b42bf45d35',
 'response_time': 2.0,
 'results': [{'content': 'Ishan Kishan was involved in a highly controversial '
                         'dismissal during an SRH vs MI IPL match, where he '
                         'surprisingly chose to walk off the',
              'raw_content': None,
              'score': 0.5746579,
              'title': 'Last time these two teams played, they created one of '
                       'the ...',
              'url': 'https://www.reddit.com/r/ipl/comments/1syhv91/last_time_these_two_teams_played_they_created_one/'},
             {'content': "One of Priyansh Arya's sixes -unfortunately the ball "
                         "landed flat on one of the spectator's face and "
                         "injured him badly. Here's wishing him a",
              'raw_content': None,
              'score': 0.5

Without Search tool

In [15]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
llm = ChatOpenAI(
    api_key = API_KEY, 
    base_url = BASE_URL,
    model = 'gpt-5.1'
).with_config(
    {'run_name': 'GPT5.1'}
)

In [16]:
from langchain.agents import create_agent

agent = create_agent(
    model = llm
)

In [17]:
question = HumanMessage(content = 'What is the updated of RCB vs KKR 2026')

resposne = agent.invoke(
    {'messages': [question]}
)

In [19]:
resposne['messages'][-1].content

'I don’t have live access to scores or future schedules, and my information only goes up to October 2024. I can’t see any 2026 match details, including RCB vs KKR.\n\nTo get the latest/updated info for an RCB vs KKR match in 2026, check:\n\n- Cricbuzz or ESPNcricinfo (live scores, commentary, schedule)\n- The official IPL website or app\n- Sports apps like SofaScore, OneCricket, or the sports section on Google (search “RCB vs KKR 2026 live score”)\n\nIf you tell me whether you’re asking about:\n- the schedule (date/venue),\n- head‑to‑head record up to now,\n- or a specific match result,\n\nI can help with historical context and stats up to 2024 and how to interpret what you see on live-score sites.'

The above model is not up to date, need to provide it a tool for live result serarch

In [21]:
from langchain.tools import tool
@tool('TavilySearch', description='Use this tool to search the web')
def web_search(query: str) -> str:
    search_tool = TavilySearch(
        max_result = 5,
        topic = 'general'
    )
    results = search_tool.invoke({'query': query})
    return results

In [22]:
live_agent = create_agent(
    model = llm, 
    tools = [web_search]
)

In [23]:
result = live_agent.invoke(
    {'messages': [question]}
)

print(result['messages'][-1].content)

Here’s the latest update for RCB vs KKR, IPL 2026 (Raipur match):

- Result: Royal Challengers Bengaluru (RCB) beat Kolkata Knight Riders (KKR) by 6 wickets.  
- Key performer: Virat Kohli scored an unbeaten century while chasing and was awarded Player of the Match.  
- Effect on table: The win took RCB to the top of the points table (at that stage of the tournament).  

If you want, I can pull the full scorecard (runs, wickets, overs, playing XI) as well.


In [24]:
for event in live_agent.stream({'messages': [question]}, stream_mode = 'updates'):
    print(event)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 140, 'total_tokens': 176, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 9, 'engine_ttft_ms': 54, 'engine_ttlt_ms': 372, 'pre_inference_ms': 142, 'service_tbt_ms': 9, 'service_ttft_ms': 415, 'service_ttlt_ms': 729, 'total_duration_ms': 597, 'user_visible_ttft_ms': 273}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DfNSa3U3laVQm2f4vc9GmxDtx3Pga', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e2600-07d6-77f1-8641-7ffa6b40e71f-0', tool_calls=[{'name': 'TavilySearch', 'args': {'query': 'RCB vs KKR 2026 match update score'}, 'id': 

In [25]:
for msg in result['messages']:
    msg.pretty_print()

================================ Human Message =================================

What is the updated of RCB vs KKR 2026
================================== Ai Message ==================================
Tool Calls:
  TavilySearch (call_ZW5fowaUpvKJaV3Mo6WCdEq2)
 Call ID: call_ZW5fowaUpvKJaV3Mo6WCdEq2
  Args:
    query: RCB vs KKR 2026 match update
================================= Tool Message =================================
Name: TavilySearch

{"query": "RCB vs KKR 2026 match update", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://sportstar.thehindu.com/cricket/ipl/rcb-vs-kkr-live-score-royal-challengers-bengaluru-kolkata-knight-riders-live-updates-ipl-2026-13-may/article70973767.ece", "title": "RCB vs KKR LIVE score, IPL 2026: Royal Challengers Bengaluru vs Kolkata Knight Riders rain updates; Match to begin at 8:45PM, toss at 8:30PM - Sportstar", "content": "# RCB vs KKR LIVE score, IPL 2026: Royal Challengers Bengaluru vs Kolkata Knight Ride